# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

print(df[['total_claim_amount', 'response']].head())

   total_claim_amount response
0          292.800000       No
1          744.924331       No
2          480.000000       No
3          484.013411      Yes
4          707.925645       No


In [2]:
# Clean the 'response' column (remove extra spaces and standardize)
df['response'] = df['response'].astype(str).str.strip().str.title()

# Create the filtered DataFrame
low_claim_yes_response = df[
    (df['total_claim_amount'] < 1000) &
    (df['response'] == 'Yes')
].copy()

# Reset index (optional)
low_claim_yes_response.reset_index(drop=True, inplace=True)

print("Filtered DataFrame (low claim + said Yes):")
print(low_claim_yes_response.shape)
print(low_claim_yes_response[['customer', 'total_claim_amount', 'response', 'gender', 'education']].head())   

Filtered DataFrame (low claim + said Yes):
(1399, 26)
  customer  total_claim_amount response gender education
0  XL78013          484.013411      Yes      M   College
1  FM55990          739.200000      Yes      M   College
2  CW49887          547.200000      Yes      F    Master
3  NJ54277           19.575683      Yes      F   College
4  MQ68407           60.036683      Yes      F  Bachelor


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [9]:
yes_customers = df[df['response'] == 'Yes']

# Simple group by policy type only (easier to understand)
simple_analysis = yes_customers.groupby('policy_type').agg({
    'monthly_premium_auto': 'mean',
    'total_claim_amount': 'mean',
    'customer': 'count'
}).round(2)

simple_analysis.columns = ['avg_premium', 'avg_claims', 'customer_count']
simple_analysis['profit'] = simple_analysis['avg_premium'] - simple_analysis['avg_claims']

print("Simple Analysis by Policy Type:")
print("\n")
print(simple_analysis)

print("\nWhich policy type is most profitable?") 
print("\n")
most_profitable = simple_analysis['profit'].idxmax()
print(f"Most profitable: {most_profitable} (${simple_analysis.loc[most_profitable, 'profit']:.2f} profit per customer)")

Simple Analysis by Policy Type:


                avg_premium  avg_claims  customer_count  profit
policy_type                                                    
Corporate Auto        93.29      421.74             323 -328.45
Personal Auto         95.06      454.98            1076 -359.92
Special Auto          89.46      441.94              67 -352.48

Which policy type is most profitable?


Most profitable: Corporate Auto ($-328.45 profit per customer)


3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [8]:
state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'customer_count']

print("All states and their customer counts:")
print("\n")
print(state_counts)

# Filter for states with more than 500 customers
popular_states = state_counts[state_counts['customer_count'] > 500]

print("\n")
print("STATES WITH 500+ CUSTOMERS:")
print("\n")
print(popular_states)

print(f"\nThere are {len(popular_states)} states with more than 500 customers")

All states and their customer counts:


        state  customer_count
0  California            3552
1      Oregon            2909
2     Arizona            1937
3      Nevada             993
4  Washington             888


STATES WITH 500+ CUSTOMERS:


        state  customer_count
0  California            3552
1      Oregon            2909
2     Arizona            1937
3      Nevada             993
4  Washington             888

There are 5 states with more than 500 customers


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [10]:
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg([
    ('max_clv', 'max'),
    ('min_clv', 'min'), 
    ('median_clv', 'median')
]).round(2).reset_index()

print(clv_stats)

              education gender   max_clv  min_clv  median_clv
0              Bachelor      F  73225.96  1904.00     5640.51
1              Bachelor      M  67907.27  1898.01     5548.03
2               College      F  61850.19  1898.68     5623.61
3               College      M  61134.68  1918.12     6005.85
4                Doctor      F  44856.11  2395.57     5332.46
5                Doctor      M  32677.34  2267.60     5577.67
6  High School or Below      F  55277.45  2144.92     6039.55
7  High School or Below      M  83325.38  1940.98     6286.73
8                Master      F  51016.07  2417.78     5729.86
9                Master      M  50568.26  2272.31     5579.10


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here